# Adult Census Income Classification
## Neural Network Approach

This notebook demonstrates the training process and results for predicting whether income exceeds $50K/yr based on census data.

## 1. Setup and Imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), 'code'))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_curve, auc

from model.neural_network import create_model
from train.data_preprocessing import DataPreprocessor
from train.trainer import Trainer

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## 2. Data Loading and Exploration

In [ ]:
# File paths
train_data_path = 'traindata.csv'
train_label_path = 'trainlabel.txt'
test_data_path = 'testdata.csv'

# Load data
preprocessor = DataPreprocessor()
train_df, train_labels, test_df = preprocessor.load_data(
    train_data_path, train_label_path, test_data_path
)

print(f"Training data shape: {train_df.shape}")
print(f"Training labels shape: {train_labels.shape}")
print(f"Test data shape: {test_df.shape}")

In [ ]:
# Display first few rows
print("\nFirst 5 rows of training data:")
train_df.head()

In [ ]:
# Basic statistics
print("Dataset Information:")
print(f"Number of features: {train_df.shape[1]}")
print(f"\nFeature names: {list(train_df.columns)}")
print(f"\nData types:")
train_df.dtypes

In [ ]:
# Label distribution
unique, counts = np.unique(train_labels, return_counts=True)
label_dist = dict(zip(unique, counts))

plt.figure(figsize=(8, 6))
plt.bar(['<=50K (0)', '>50K (1)'], [label_dist[0], label_dist[1]], color=['#3498db', '#e74c3c'])
plt.ylabel('Count')
plt.title('Distribution of Income Labels in Training Data')
plt.grid(axis='y', alpha=0.3)
for i, (label, count) in enumerate(label_dist.items()):
    plt.text(i, count, f'{count}\n({count/sum(counts)*100:.1f}%)', 
             ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/label_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Class distribution:")
print(f"  <=50K (0): {label_dist[0]} ({label_dist[0]/sum(counts)*100:.2f}%)")
print(f"  >50K (1): {label_dist[1]} ({label_dist[1]/sum(counts)*100:.2f}%)")

In [ ]:
# Check for missing values
missing_train = train_df.isna().sum()
missing_test = test_df.isna().sum()

# Check for ' ?' values
question_train = (train_df == ' ?').sum()
question_test = (test_df == ' ?').sum()

print("Missing values (NaN) in training data:")
print(missing_train[missing_train > 0])
print("\n' ?' values in training data:")
print(question_train[question_train > 0])

## 3. Data Preprocessing

In [ ]:
# Preprocess data
print("Preprocessing data...")
X_train_scaled, y_train, X_test_scaled, feature_names = preprocessor.preprocess(
    train_df, train_labels, test_df
)

print(f"\nPreprocessed training data shape: {X_train_scaled.shape}")
print(f"Preprocessed test data shape: {X_test_scaled.shape}")
print(f"Number of features after preprocessing: {len(feature_names)}")

In [ ]:
# Split training data
X_train, X_val, y_train_split, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print(f"Test set size: {X_test_scaled.shape[0]}")

## 4. Model Architecture

In [ ]:
# Create modeldevice = 'cuda' if torch.cuda.is_available() else 'cpu'input_dim = X_train.shape[1]model = create_model(    input_dim=input_dim,    hidden_dims=[128, 64, 32])print("Neural Network Architecture:")print("="*50)print(model)print("="*50)# Count parameterstotal_params = sum(p.numel() for p in model.parameters())trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)print(f"\nTotal parameters: {total_params:,}")print(f"Trainable parameters: {trainable_params:,}")

## 5. Model Training

In [ ]:
# Create trainer
trainer = Trainer(model, device=device, learning_rate=0.001)

# Train model
print("Training the model...")
history = trainer.fit(
    X_train, y_train_split,
    X_val, y_val,
    epochs=50,
    batch_size=128,
    verbose=True
)

print("\nTraining completed!")

## 6. Training Results Visualization

In [ ]:
# Plot training history
os.makedirs('figures', exist_ok=True)
trainer.plot_training_history(save_path='figures/training_history.png')
plt.show()

## 7. Model Evaluation

In [ ]:
# Evaluate on validation set
metrics, val_predictions = trainer.evaluate(X_val, y_val)

print("Validation Set Metrics:")
print("="*50)
for metric_name, value in metrics.items():
    print(f"{metric_name.capitalize():20s}: {value:.4f}")
print("="*50)

In [ ]:
# Confusion Matrix
trainer.plot_confusion_matrix(y_val, val_predictions, save_path='figures/confusion_matrix.png')
plt.show()

In [ ]:
# Classification Report
print("\nDetailed Classification Report:")
print(classification_report(y_val, val_predictions, 
                          target_names=['<=50K', '>50K'],
                          digits=4))

In [ ]:
# ROC Curve
val_probabilities = trainer.predict_proba(X_val)
fpr, tpr, thresholds = roc_curve(y_val, val_probabilities)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figures/roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Test Set Predictions

In [ ]:
# Make predictions on test set
test_predictions = trainer.predict(X_test_scaled)

print(f"Test set predictions generated: {len(test_predictions)} samples")

# Prediction distribution
unique, counts = np.unique(test_predictions, return_counts=True)
print("\nTest Set Prediction Distribution:")
for label, count in zip(unique, counts):
    class_name = '<=50K' if label == 0 else '>50K'
    print(f"  {class_name} ({label}): {count} ({count/len(test_predictions)*100:.2f}%)")

In [ ]:
# Visualize test predictions
plt.figure(figsize=(8, 6))
plt.bar(['<=50K (0)', '>50K (1)'], counts, color=['#3498db', '#e74c3c'])
plt.ylabel('Count')
plt.title('Distribution of Predictions on Test Data')
plt.grid(axis='y', alpha=0.3)
for i, count in enumerate(counts):
    plt.text(i, count, f'{count}\n({count/len(test_predictions)*100:.1f}%)', 
             ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/test_predictions_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Save Predictions

In [ ]:
# Save predictions to file
output_path = 'testlabel.txt'
with open(output_path, 'w') as f:
    for pred in test_predictions:
        f.write(f"{pred}\n")

print(f"✓ Predictions saved to: {output_path}")

## 10. Summary Statistics for Report

In [ ]:
# Create summary table
summary_data = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Value': [
        f"{metrics['accuracy']:.4f}",
        f"{metrics['precision']:.4f}",
        f"{metrics['recall']:.4f}",
        f"{metrics['f1_score']:.4f}",
        f"{metrics['roc_auc']:.4f}"
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\nModel Performance Summary:")
print(summary_df.to_string(index=False))

# Save to CSV for LaTeX table
summary_df.to_csv('figures/performance_metrics.csv', index=False)
print("\n✓ Performance metrics saved to figures/performance_metrics.csv")

In [ ]:
# Model configuration summary
config_data = {
    'Parameter': ['Input Dimensions', 'Hidden Layers', 'Dropout Rate', 
                  'Learning Rate', 'Batch Size', 'Epochs', 'Optimizer'],
    'Value': [
        f"{input_dim}",
        "[128, 64, 32]",
        "0.3",
        "0.001",
        "128",
        "50",
        "Adam"
    ]
}

config_df = pd.DataFrame(config_data)
print("\nModel Configuration:")
print(config_df.to_string(index=False))

config_df.to_csv('figures/model_configuration.csv', index=False)
print("\n✓ Model configuration saved to figures/model_configuration.csv")